In [ ]:
import keras
from keras import ops
from keras import layers

import tensorflow as tf
from tensorflow.keras import backend
import random

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from scipy.fft import fft, ifft, fftfreq
import pywt
import gc
import warnings
import os

In [ ]:
## Code adapted from: https://keras.io/examples/nlp/text_classification_with_transformer/
## Additional codes modified from https://github.com/samugit83/TheGradientPath/blob/master/Keras/transformers/time_series_forecast/main.py


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=maxlen, step=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerEncoder, self).__init__()
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate)
                           for _ in range(num_layers)]
        self.dropout = layers.Dropout(rate)

    def call(self, inputs, training=False):
        x = inputs
        x = self.dropout(x, training=training)
        for layer in self.enc_layers:
            x = layer(x, training=training)
        return x

In [ ]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data.iloc[i:(i + time_step),:])
        y.append(data.iloc[i + time_step,:])
    X, y = np.array(X), np.array(y)
    # y = y.flatten()
    return X, y

In [ ]:
def model_builder(maxlen, embed_dim, num_heads, ff_dim, dim):
  inputs = layers.Input(shape=(maxlen,dim))
  x = layers.Dense(embed_dim)(inputs)
  encoder = TransformerEncoder(num_layers=8, embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim, rate=0.1)
  x = encoder(x)
  transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
  x = transformer_block(x)
  x = tf.keras.layers.Flatten()(x)
  # x = layers.GlobalAveragePooling1D()(x)
  # x = layers.Dropout(0.1)(x)
  # x = layers.Dense(64, activation="relu")(x)
  x = layers.Dropout(0.1)(x)
  x = layers.Dense(64, activation="relu")(x)
  x = layers.Dropout(0.1)(x)
  outputs = layers.Dense(dim)(x) #layers.Dense(2, activation="softmax")(x)

  model = keras.Model(inputs=inputs, outputs=outputs)
  return model

## Fine-Tuning Code:

In [ ]:
def base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, seed_nr, epochs, plot_yes_no):
  random.seed(seed_nr)
  np.random.seed(seed_nr)
  tf.random.set_seed(seed_nr)
  model = model_builder(maxlen, embed_dim, num_heads, ff_dim, y.shape[1])
  X_train = X[:-test_len - valid_len]
  y_train = y[:-test_len - valid_len]

  X_valid = X[-test_len - valid_len:-test_len]
  y_valid = y[-test_len - valid_len:-test_len]

  X_test = X[-test_len:]
  y_test = y[-test_len:]

  model.compile(optimizer="adam", loss="mse", metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')])
  model.fit(X_train, y_train, batch_size=32, epochs=epochs, validation_data=(X_valid, y_valid))
  model.save_weights('model_wt.weights.h5')

  stock_data = hist[cols_list]

  prediction_short = model.predict(X)
  prediction_short_df = pd.DataFrame(prediction_short, columns = cols_list)
  prediction_short_df.index = stock_data[-len(prediction_short_df):].index
  prediction_short_df_rescaled = prediction_short_df * (max_list - min_list) + min_list

  NRMSE_short_list = []
  for i in range(len(cols_list)):
      rmse_value_short = np.sqrt(np.mean(((prediction_short_df_rescaled[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]) ** 2)))
      nrmse_value_short = rmse_value_short/(stock_data[cols_list[i]].iloc[-test_len:].max()-stock_data[cols_list[i]].iloc[-test_len:].min())
      NRMSE_short_list.append(float(nrmse_value_short))

      if plot_yes_no == 'Yes':
        # Plotting the rescaled data:
        plt.figure(figsize=(16,6))
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[:-test_len - valid_len], 'm-', label = 'Training')
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[-test_len - valid_len:-test_len], 'g-', label = 'Validation')
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[-test_len:], 'r-', label = '1-Day Prediction')
        plt.plot(hist[cols_list[i]], 'b-', label = 'Actual')
        plt.title(f'{stock_symbol} ({cols_list[i]})', fontsize=12)
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.show()
  print('NRMSE (Base): ', NRMSE_short_list)
  backend.clear_session(free_memory = True)
  del model
  gc.collect()

In [ ]:
def transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, seed_nr, epochs, plot_yes_no):
  random.seed(seed_nr)
  np.random.seed(seed_nr)
  tf.random.set_seed(seed_nr)
  model = model_builder(maxlen, embed_dim, num_heads, ff_dim, y.shape[1])

  X_train = X[:-test_len - valid_len]
  y_train = y[:-test_len - valid_len]

  X_valid = X[-test_len - valid_len:-test_len]
  y_valid = y[-test_len - valid_len:-test_len]

  X_test = X[-test_len:]
  y_test = y[-test_len:]

  stock_data = hist[cols_list]


  model.load_weights('model_wt.weights.h5')
  model.compile(optimizer="adam", loss="mse", metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

  model.fit(X_train, y_train, batch_size=32, epochs=epochs, validation_data=(X_valid, y_valid))

  prediction_short = model.predict(X)
  prediction_short_df = pd.DataFrame(prediction_short, columns = cols_list)
  prediction_short_df.index = stock_data[-len(prediction_short_df):].index
  prediction_short_df_rescaled = prediction_short_df * (max_list - min_list) + min_list

  prediction_long = model.predict(X[:-test_len])
  prediction_long_list = prediction_long.tolist()

  prediction_test_val_rescaled = prediction_long* (max_list[time_step+1:-test_len].values - min_list[time_step+1:-test_len].values) + min_list[time_step+1:-test_len].values
  prediction_test_val_rescaled = prediction_test_val_rescaled.tolist()

  prediction_stock_rescaled = stock_data[:-test_len].values.tolist()
  X_test_singular = X[-test_len:-test_len+1]
  prediction_test_long_rescaled = []
  for _ in range(test_len):
    predicted_value = model.predict(X_test_singular)

    prediction_long_list.append(predicted_value[0].tolist())

    X_test_singular = np.append(X_test_singular[0,1:],predicted_value, axis=0)
    X_test_singular = np.reshape(X_test_singular, (1, X_test_singular.shape[0],
                                              X_test_singular.shape[1]))

    min_val = np.min(prediction_stock_rescaled[-100:], axis=0)
    max_val = np.max(prediction_stock_rescaled[-100:], axis=0)

    predicted_value_rescaled = predicted_value*(max_val-min_val) + min_val
    prediction_test_long_rescaled.append(predicted_value_rescaled[0].tolist())
    prediction_stock_rescaled.append(predicted_value_rescaled[0].tolist())

  prediction_long_df = pd.DataFrame(prediction_long_list, columns = cols_list)
  prediction_long_df.index = stock_data[-len(prediction_long_df):].index

  prediction_long_rescaled = prediction_test_val_rescaled + prediction_test_long_rescaled
  prediction_long_df_rescaled = pd.DataFrame(prediction_long_rescaled, columns = cols_list)
  prediction_long_df_rescaled.index = stock_data[-len(prediction_long_df_rescaled):].index

  stock_data

  NRMSE_short_list = []
  NRMSE_long_list = []
  for i in range(len(cols_list)):
      rmse_value_short = np.sqrt(np.mean(((prediction_short_df_rescaled[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]) ** 2)))
      nrmse_value_short = rmse_value_short/(stock_data[cols_list[i]].iloc[-test_len:].max()-stock_data[cols_list[i]].iloc[-test_len:].min())
      NRMSE_short_list.append(float(nrmse_value_short))

      rmse_value_long = np.sqrt(np.mean(((prediction_long_df_rescaled[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]) ** 2)))
      nrmse_value_long = rmse_value_long/(stock_data[cols_list[i]].iloc[-test_len:].max()-stock_data[cols_list[i]].iloc[-test_len:].min())
      NRMSE_long_list.append(float(nrmse_value_long))

      if plot_yes_no == 'Yes':
        ## Plotting the unscaled data:
        plt.figure(figsize=(16,6))
        # plt.plot(prediction_df[cols_list[i]], 'r-', label = 'Prediction')
        plt.plot(prediction_short_df.index, y[:,i], 'b-', label = 'Actual')
        plt.plot(prediction_short_df[cols_list[i]].iloc[:-test_len - valid_len], 'm-', label = 'Training')
        plt.plot(prediction_short_df[cols_list[i]].iloc[-test_len - valid_len:-test_len], 'g-', label = 'Validation')
        plt.plot(prediction_short_df[cols_list[i]].iloc[-test_len:], 'r-', label = '1-Day Prediction')
        plt.plot(prediction_long_df[cols_list[i]].iloc[-test_len:], color='firebrick', linestyle = '-', label = f'{test_len}-Day Prediction')
        plt.title(f'{stock_symbol} ({cols_list[i]}) - Unscaled', fontsize=12)
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.show()

        ## Plotting the rescaled data:
        plt.figure(figsize=(16,6))
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[:-test_len - valid_len], 'm-', label = 'Training')
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[-test_len - valid_len:-test_len], 'g-', label = 'Validation')
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[-test_len:], 'r-', label = '1-Day Prediction')
        plt.plot(prediction_long_df_rescaled[cols_list[i]].iloc[-test_len:], color='firebrick', linestyle = '-', label = f'{test_len}-Day Prediction')
        plt.plot(stock_data[cols_list[i]], 'b-', label = 'Actual')
        plt.title(f'{stock_symbol} ({cols_list[i]})', fontsize=12)
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.show()
  backend.clear_session(free_memory = True)
  del model
  gc.collect()
  return NRMSE_short_list, NRMSE_long_list

## Fourier Denoising functions:

In [ ]:
## Padding-based Fourier Transform, based on Song et al. (2021):
def padding_fourier(stock_data, quantile_percent, nr_padding_samples):
  denoised_stock_data = pd.DataFrame()

  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]].reset_index(drop=True)

    ## Creating the padding:
    X1_bar = np.sqrt(np.mean(xt[:20]))
    X2_bar = np.sqrt(np.mean(xt[-20:]))

    sigma1 = np.sqrt(np.mean(xt[:20]-X1_bar))
    sigma2 = np.sqrt(np.mean(xt[-20:]-X2_bar))

    np.random.seed(i)
    N1_list = np.random.normal(loc=0, scale=sigma1, size=nr_padding_samples)
    N2_list = np.random.normal(loc=0, scale=sigma2, size=nr_padding_samples)

    lower_pad_series = pd.Series(np.flip(N1_list.cumsum())+xt.iloc[0])
    upper_pad_series = pd.Series(N1_list.cumsum()+xt.iloc[-1])

    xt = pd.concat([lower_pad_series, xt, upper_pad_series], ignore_index=True)

    fxt = np.fft.fftn(xt)
    if stock_data.columns[i] == 'Volume':
      threshold_value = np.quantile(np.abs(fxt), [0.99])
    else:
      threshold_value = np.quantile(np.abs(fxt), [quantile_percent])
    index = abs(fxt)> threshold_value[0] #list[i]
    fxt_clean = fxt*index.T
    xt_clean = np.fft.ifftn(fxt_clean)

    xt_unpadded = xt_clean[nr_padding_samples:-nr_padding_samples]
    denoised_stock_data[stock_data.columns[i]] = xt_unpadded.real

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
def fourier_window_renoising(stock_data, quantile_percent, window_coeff):
  denoised_stock_data = pd.DataFrame()

  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]]
    fxt = np.fft.fftn(xt)
    if stock_data.columns[i] == 'Volume':
      threshold_value = np.quantile(np.abs(fxt), [0.99])
    else:
      threshold_value = np.quantile(np.abs(fxt), [quantile_percent])
    index = abs(fxt)> threshold_value[0]
    fxt_clean = fxt*index.T
    xt_clean = np.fft.ifftn(fxt_clean)

    index_renoise = abs(fxt)<= threshold_value[0]
    fxt_renoise = fxt*index_renoise.T
    xt_renoise = np.fft.ifftn(fxt_renoise)
    if stock_data.columns[i] == 'Volume':
      a = 20/len(xt_renoise)**2
    else:
      a = window_coeff/len(xt_renoise)**2
    date_range = np.linspace(0,len(stock_data)-1,len(stock_data))
    renoise_window = np.maximum(0,a*date_range*(date_range-len(xt_renoise))+1)
    fwr_data = xt_clean+xt_renoise*renoise_window
    denoised_stock_data[stock_data.columns[i]] = fwr_data.real

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data



In [ ]:
def linear_drift_denoising(stock_data, quantile_percent):
  denoised_stock_data = pd.DataFrame()

  date_range = np.linspace(0,len(stock_data)-1,len(stock_data))
  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]]
    lin_drift_slope = (xt.iloc[-1] - xt.iloc[0])/len(xt)
    lin_drift = []
    for j in date_range:
      lin_drift.append(float(xt.iloc[0] + lin_drift_slope * j))
    xt_undrifted = xt - lin_drift
    fxt = np.fft.fft(xt_undrifted)
    if stock_data.columns[i] == 'Volume':
      threshold_value = np.quantile(np.abs(fxt), [0.99])
    else:
      threshold_value = np.quantile(np.abs(fxt), [quantile_percent])
    index = abs(fxt)> threshold_value[0]
    fxt_clean = fxt*index
    xt_clean = np.fft.ifft(fxt_clean)
    xt_clean = xt_clean.real
    xt_clean_redrifted = xt_clean + lin_drift

    denoised_stock_data[stock_data.columns[i]] = xt_clean_redrifted

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
def stationary_denoising(stock_data, quantile_percent, min_max_range, smoothing_range):
  ## -------------- Creating the variable Min-Max range: ----------------------
  min_val = stock_data.head(min_max_range).min().tolist()
  max_val = stock_data.head(min_max_range).max().tolist()

  min_list_vals = [min_val[:] for _ in range(min_max_range)]
  max_list_vals = [max_val[:] for _ in range(min_max_range)]

  # test_len

  for i in range(len(stock_data)-min_max_range):
    min_val = stock_data.iloc[i:i+min_max_range,:].min().tolist()
    max_val = stock_data.iloc[i:i+min_max_range,:].max().tolist()
    min_list_vals.append(min_val)
    max_list_vals.append(max_val)

  # min_list = pd.DataFrame()
  # max_list = pd.DataFrame()

  min_list = pd.DataFrame(min_list_vals, columns = stock_data.columns.tolist())
  max_list = pd.DataFrame(max_list_vals, columns = stock_data.columns.tolist())

  min_list.index = stock_data.index
  max_list.index = stock_data.index

  ## Setting fixed scaling range for Volume to improve consistency:
  min_list['Volume'] = stock_data['Volume'].min()
  max_list['Volume'] = stock_data['Volume'].max()

  avg_list = (min_list+max_list)/2

  data_scaled = (stock_data - avg_list)/(max_list - min_list)

  ## -------- Smoothing the minmax scaling: --------------
  min_list_smoothed = min_list.iloc[:smoothing_range,:].values.tolist()
  max_list_smoothed = max_list.iloc[:smoothing_range,:].values.tolist()

  for i in range(len(min_list)-int(2*smoothing_range)):
    min_vals_smoothed = min_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    min_list_smoothed.append(min_vals_smoothed)

    max_vals_smoothed = max_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    max_list_smoothed.append(max_vals_smoothed)

  end_vals_min = min_list.iloc[-smoothing_range:,:].values.tolist()
  min_list_smoothed = min_list_smoothed + end_vals_min

  end_vals_max = max_list.iloc[-smoothing_range:,:].values.tolist()
  max_list_smoothed = max_list_smoothed + end_vals_max

  min_list_smoothed = pd.DataFrame(min_list_smoothed, columns = stock_data.columns.tolist())
  min_list_smoothed.index = min_list.index
  min_list_smoothed

  max_list_smoothed = pd.DataFrame(max_list_smoothed, columns = stock_data.columns.tolist())
  max_list_smoothed.index = max_list.index
  max_list_smoothed

  avg_list_smoothed = (min_list_smoothed+max_list_smoothed)/2

  ## FFT denoising of smoothed curve:
  denoised_stock_data_variable = pd.DataFrame()
  for i in range(len(data_scaled.columns)):
    ## Creating the FFT smoothed curve:
    xt = data_scaled[data_scaled.columns[i]]
    fxt = np.fft.fftn(xt)

    if stock_data.columns[i] == 'Volume':
      threshold_value = np.quantile(np.abs(fxt), [0.99])
    else:
      threshold_value = np.quantile(np.abs(fxt), [quantile_percent])
    index = abs(fxt)> threshold_value[0]
    fxt_clean = fxt*index.T

    xt_clean = np.fft.ifftn(fxt_clean)
    xt_clean = xt_clean.real
    denoised_stock_data_variable[data_scaled.columns[i]] = xt_clean

  denoised_stock_data_variable.index = stock_data.index

  ## ------------- Rescaling using the smoothed Min-Max scaling: ----------------

  # denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list - min_list) + avg_list
  denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list_smoothed - min_list_smoothed) + avg_list_smoothed

  return denoised_stock_data_rescaled

In [ ]:
def stationary_fejer_denoising(stock_data, quantile_percent, min_max_range, smoothing_range):
  ## -------------- Creating the variable Min-Max range: ----------------------
  min_val = stock_data.head(min_max_range).min().tolist()
  max_val = stock_data.head(min_max_range).max().tolist()

  min_list_vals = [min_val[:] for _ in range(min_max_range)]
  max_list_vals = [max_val[:] for _ in range(min_max_range)]

  # test_len

  for i in range(len(stock_data)-min_max_range):
    min_val = stock_data.iloc[i:i+min_max_range,:].min().tolist()
    max_val = stock_data.iloc[i:i+min_max_range,:].max().tolist()
    min_list_vals.append(min_val)
    max_list_vals.append(max_val)

  # min_list = pd.DataFrame()
  # max_list = pd.DataFrame()

  min_list = pd.DataFrame(min_list_vals, columns = stock_data.columns.tolist())
  max_list = pd.DataFrame(max_list_vals, columns = stock_data.columns.tolist())

  min_list.index = stock_data.index
  max_list.index = stock_data.index

  ## Setting fixed scaling range for Volume to improve consistency:
  min_list['Volume'] = stock_data['Volume'].min()
  max_list['Volume'] = stock_data['Volume'].max()

  avg_list = (min_list+max_list)/2

  data_scaled = (stock_data - avg_list)/(max_list - min_list)

  ## -------- Smoothing the minmax scaling: --------------
  min_list_smoothed = min_list.iloc[:smoothing_range,:].values.tolist()
  max_list_smoothed = max_list.iloc[:smoothing_range,:].values.tolist()

  for i in range(len(min_list)-int(2*smoothing_range)):
    min_vals_smoothed = min_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    min_list_smoothed.append(min_vals_smoothed)

    max_vals_smoothed = max_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    max_list_smoothed.append(max_vals_smoothed)

  end_vals_min = min_list.iloc[-smoothing_range:,:].values.tolist()
  min_list_smoothed = min_list_smoothed + end_vals_min

  end_vals_max = max_list.iloc[-smoothing_range:,:].values.tolist()
  max_list_smoothed = max_list_smoothed + end_vals_max

  min_list_smoothed = pd.DataFrame(min_list_smoothed, columns = stock_data.columns.tolist())
  min_list_smoothed.index = min_list.index
  min_list_smoothed

  max_list_smoothed = pd.DataFrame(max_list_smoothed, columns = stock_data.columns.tolist())
  max_list_smoothed.index = max_list.index
  max_list_smoothed

  avg_list_smoothed = (min_list_smoothed+max_list_smoothed)/2

  ## FFT denoising of smoothed curve:
  denoised_stock_data_variable = pd.DataFrame()
  sk_df = pd.DataFrame()
  Cesaro_df = pd.DataFrame()
  warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
  for i in range(len(data_scaled.columns)):
    xt = data_scaled[data_scaled.columns[i]]
    fxt = np.fft.fftn(xt)
    fxt = np.fft.fftshift(fxt)

    index_mtx = np.tril(np.ones((len(fxt), len(fxt))))
    sk_df['PS0'] = np.zeros(len(xt))
    for j in range(len(fxt)):
      index_fej = index_mtx[j].tolist()
      fxt_singular = fxt.T * index_fej
      fxt_singular = fxt_singular.T
      fxt_singular = np.fft.ifftshift(fxt_singular)
      xt_singular = np.fft.ifftn(fxt_singular)
      xt_singular = xt_singular.real.T
      sk_df['PS'+str(j+1)] = xt_singular

    for j in range(len(fxt)):
      col_list = sk_df.columns[:j+1].tolist()
      Cesaro_df['CM'+str(j+1)] = sk_df[col_list].mean(axis=1)

    denoised_stock_data_variable[data_scaled.columns[i]] = Cesaro_df.iloc[:,-1]
  denoised_stock_data_variable.index = stock_data.index

  ## ------------- Rescaling using the smoothed Min-Max scaling: ----------------

  # denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list - min_list) + avg_list
  denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list_smoothed - min_list_smoothed) + avg_list_smoothed

  return denoised_stock_data_rescaled

In [ ]:
def stationary_lanczos_denoising(stock_data, quantile_percent, min_max_range, smoothing_range):
  ## -------------- Creating the variable Min-Max range: ----------------------
  min_val = stock_data.head(min_max_range).min().tolist()
  max_val = stock_data.head(min_max_range).max().tolist()

  min_list_vals = [min_val[:] for _ in range(min_max_range)]
  max_list_vals = [max_val[:] for _ in range(min_max_range)]

  # test_len

  for i in range(len(stock_data)-min_max_range):
    min_val = stock_data.iloc[i:i+min_max_range,:].min().tolist()
    max_val = stock_data.iloc[i:i+min_max_range,:].max().tolist()
    min_list_vals.append(min_val)
    max_list_vals.append(max_val)

  # min_list = pd.DataFrame()
  # max_list = pd.DataFrame()

  min_list = pd.DataFrame(min_list_vals, columns = stock_data.columns.tolist())
  max_list = pd.DataFrame(max_list_vals, columns = stock_data.columns.tolist())

  min_list.index = stock_data.index
  max_list.index = stock_data.index

  ## Setting fixed scaling range for Volume to improve consistency:
  min_list['Volume'] = stock_data['Volume'].min()
  max_list['Volume'] = stock_data['Volume'].max()

  avg_list = (min_list+max_list)/2

  data_scaled = (stock_data - avg_list)/(max_list - min_list)

  ## -------- Smoothing the minmax scaling: --------------
  min_list_smoothed = min_list.iloc[:smoothing_range,:].values.tolist()
  max_list_smoothed = max_list.iloc[:smoothing_range,:].values.tolist()

  for i in range(len(min_list)-int(2*smoothing_range)):
    min_vals_smoothed = min_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    min_list_smoothed.append(min_vals_smoothed)

    max_vals_smoothed = max_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    max_list_smoothed.append(max_vals_smoothed)

  end_vals_min = min_list.iloc[-smoothing_range:,:].values.tolist()
  min_list_smoothed = min_list_smoothed + end_vals_min

  end_vals_max = max_list.iloc[-smoothing_range:,:].values.tolist()
  max_list_smoothed = max_list_smoothed + end_vals_max

  min_list_smoothed = pd.DataFrame(min_list_smoothed, columns = stock_data.columns.tolist())
  min_list_smoothed.index = min_list.index
  min_list_smoothed

  max_list_smoothed = pd.DataFrame(max_list_smoothed, columns = stock_data.columns.tolist())
  max_list_smoothed.index = max_list.index
  max_list_smoothed

  avg_list_smoothed = (min_list_smoothed+max_list_smoothed)/2

  ## FFT denoising of smoothed curve:
  denoised_stock_data_variable = pd.DataFrame()
  for i in range(len(data_scaled.columns)):
    ## Creating the FFT smoothed curve:
    xt = data_scaled[data_scaled.columns[i]]
    fxt = np.fft.fftn(xt)
    sinc_factor = np.sin(np.pi*np.arange(len(fxt))/len(fxt))/(np.pi*np.arange(len(fxt))/len(fxt))
    sinc_factor[0] = 1

    fxt_clean = fxt*sinc_factor

    xt_clean = np.fft.ifftn(fxt_clean)
    xt_clean = xt_clean.real
    denoised_stock_data_variable[data_scaled.columns[i]] = xt_clean

  denoised_stock_data_variable.index = stock_data.index

  ## ------------- Rescaling using the smoothed Min-Max scaling: ----------------

  # denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list - min_list) + avg_list
  denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list_smoothed - min_list_smoothed) + avg_list_smoothed

  return denoised_stock_data_rescaled

In [ ]:
def stationary_exponential_denoising(stock_data, quantile_percent, min_max_range, smoothing_range, exponential_decay_rate):
  ## -------------- Creating the variable Min-Max range: ----------------------
  min_val = stock_data.head(min_max_range).min().tolist()
  max_val = stock_data.head(min_max_range).max().tolist()

  min_list_vals = [min_val[:] for _ in range(min_max_range)]
  max_list_vals = [max_val[:] for _ in range(min_max_range)]

  # test_len

  for i in range(len(stock_data)-min_max_range):
    min_val = stock_data.iloc[i:i+min_max_range,:].min().tolist()
    max_val = stock_data.iloc[i:i+min_max_range,:].max().tolist()
    min_list_vals.append(min_val)
    max_list_vals.append(max_val)

  # min_list = pd.DataFrame()
  # max_list = pd.DataFrame()

  min_list = pd.DataFrame(min_list_vals, columns = stock_data.columns.tolist())
  max_list = pd.DataFrame(max_list_vals, columns = stock_data.columns.tolist())

  min_list.index = stock_data.index
  max_list.index = stock_data.index

  ## Setting fixed scaling range for Volume to improve consistency:
  min_list['Volume'] = stock_data['Volume'].min()
  max_list['Volume'] = stock_data['Volume'].max()

  avg_list = (min_list+max_list)/2

  data_scaled = (stock_data - avg_list)/(max_list - min_list)

  ## -------- Smoothing the minmax scaling: --------------
  min_list_smoothed = min_list.iloc[:smoothing_range,:].values.tolist()
  max_list_smoothed = max_list.iloc[:smoothing_range,:].values.tolist()

  for i in range(len(min_list)-int(2*smoothing_range)):
    min_vals_smoothed = min_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    min_list_smoothed.append(min_vals_smoothed)

    max_vals_smoothed = max_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    max_list_smoothed.append(max_vals_smoothed)

  end_vals_min = min_list.iloc[-smoothing_range:,:].values.tolist()
  min_list_smoothed = min_list_smoothed + end_vals_min

  end_vals_max = max_list.iloc[-smoothing_range:,:].values.tolist()
  max_list_smoothed = max_list_smoothed + end_vals_max

  min_list_smoothed = pd.DataFrame(min_list_smoothed, columns = stock_data.columns.tolist())
  min_list_smoothed.index = min_list.index
  min_list_smoothed

  max_list_smoothed = pd.DataFrame(max_list_smoothed, columns = stock_data.columns.tolist())
  max_list_smoothed.index = max_list.index
  max_list_smoothed

  avg_list_smoothed = (min_list_smoothed+max_list_smoothed)/2

  ## FFT denoising of smoothed curve:
  denoised_stock_data_variable = pd.DataFrame()
  for i in range(len(data_scaled.columns)):
    ## Creating the FFT smoothed curve:
    xt = data_scaled[data_scaled.columns[i]]
    fxt = np.fft.fftn(xt)

    exponential_decay_factor = exponential_decay_rate**(np.arange(len(fxt)))
    fxt_clean = fxt*exponential_decay_factor

    xt_clean = np.fft.ifftn(fxt_clean)
    xt_clean = xt_clean.real
    denoised_stock_data_variable[data_scaled.columns[i]] = xt_clean

  denoised_stock_data_variable.index = stock_data.index

  ## ------------- Rescaling using the smoothed Min-Max scaling: ----------------

  # denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list - min_list) + avg_list
  denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list_smoothed - min_list_smoothed) + avg_list_smoothed

  return denoised_stock_data_rescaled

In [ ]:
def stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no):
  test_len = 30
  memory = 20

  stock = yf.Ticker(stock_symbol)
  hist = stock.history(start=start_date, end=end_date, interval='1d')
  # hist.to_csv(f'{stock_symbol}_{start_date}_{end_date}.csv')
  stock_data = hist[cols_list]

  if denoising_method == 'None':
    denoised_stock_data = stock_data
  elif denoising_method == 'Padding':
    nr_padding_samples = 20
    denoised_stock_data = padding_fourier(stock_data.iloc[:-test_len], quantile_percent, nr_padding_samples)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'FWR':
    window_coeff = 4
    denoised_stock_data = fourier_window_renoising(stock_data.iloc[:-test_len], quantile_percent, window_coeff)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'LinDrift':
    denoised_stock_data = linear_drift_denoising(stock_data.iloc[:-test_len], quantile_percent)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Stationary':
    min_max_range = 5
    smoothing_range = 5
    denoised_stock_data = stationary_denoising(stock_data.iloc[:-test_len], quantile_percent, min_max_range, smoothing_range)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Fejer':
    min_max_range = 5
    smoothing_range = 5
    denoised_stock_data = stationary_fejer_denoising(stock_data[:-test_len], quantile_percent, min_max_range, smoothing_range)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Lanczos':
    min_max_range = 5
    smoothing_range = 5
    denoised_stock_data = stationary_lanczos_denoising(stock_data[:-test_len], quantile_percent, min_max_range, smoothing_range)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Exponential':
    min_max_range = 5
    smoothing_range = 5
    exponential_decay_rate = 0.9
    denoised_stock_data = stationary_exponential_denoising(stock_data[:-test_len], quantile_percent, min_max_range, smoothing_range, exponential_decay_rate)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])



  if denoising_method != 'None' and plot_yes_no == 'Yes':
    plt.figure(figsize=(16,int(6*len(cols_list))))
    i = 0
    for col in cols_list:
      i = i+1
      plt.subplot(len(stock_data.columns), 1, i)
      plt.xlabel('Date')
      plt.ylabel(col)
      plt.plot(stock_data[col], color = 'b', linestyle = '-', label = f'Original ({col})')
      plt.plot(denoised_stock_data[col], color = 'peru', linestyle = '-', label = f'Denoised ({col})')
      plt.legend()
    plt.show()


  ## ---------- Stationary signal scaling for transformer: -------------
  min_max_len = 50

  min_val = denoised_stock_data.head(min_max_len).min().tolist()
  max_val = denoised_stock_data.head(min_max_len).max().tolist()

  min_list_vals = [min_val[:] for _ in range(min_max_len)]
  max_list_vals = [max_val[:] for _ in range(min_max_len)]

  for i in range(len(denoised_stock_data)-min_max_len):
    min_val = denoised_stock_data.iloc[i:i+min_max_len,:].min().tolist()
    max_val = denoised_stock_data.iloc[i:i+min_max_len,:].max().tolist()
    min_list_vals.append(min_val)
    max_list_vals.append(max_val)


  min_list = pd.DataFrame()
  max_list = pd.DataFrame()

  min_list = pd.DataFrame(min_list_vals, columns = cols_list)
  max_list = pd.DataFrame(max_list_vals, columns = cols_list)

  if 'Volume' in cols_list:
    min_list['Volume'] = hist['Volume'].min()
    max_list['Volume'] = hist['Volume'].max()

  min_list.index = denoised_stock_data.index
  max_list.index = denoised_stock_data.index

  data_scaled = (stock_data - min_list)/(max_list - min_list)
  data_scaled

  ## ---------------- Creating training and testing data: -----------------------
  X, y = create_dataset(data_scaled, time_step=time_step)

  return hist, X, y, min_list, max_list


In [ ]:
## Stocks: ['^GSPTSE', 'SLV', 'MSFT', 'TSLA', 'CNQ.TO', 'BHP', 'BMO', 'V', 'L.TO', 'USO']

stock_symbol_list = ['^GSPTSE']

start_date_list = ['2022-01-01'] #['2022-01-01', '2022-09-01', '2023-05-01', '2024-01-01']

end_date_list = ['2024-01-01'] #['2024-01-01', '2024-09-01', '2025-05-01', '2026-01-01']

cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

## Transformer:

In [ ]:
denoising_method = 'None'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()

## Padding-Based Fourier Denoising

Based on Song et al. (2021):

In [ ]:
denoising_method = 'Padding'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()

## Fourier Window Renoising:

In [ ]:
denoising_method = 'FWR'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()

## Fourier Linear Drift Denoising:

In [ ]:
denoising_method = 'LinDrift'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()

## Stationary Signal Denoising:

In [ ]:
denoising_method = 'Stationary'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()

## Fejer Summation Denoising:

In [ ]:
denoising_method = 'Fejer'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()

## Lanczos $\sigma$-Approximation:

In [ ]:
denoising_method = 'Lanczos'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()

## Stationary Exponential Denoising:

In [ ]:
denoising_method = 'Exponential'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'



method_used = '' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_NRMSE.csv') and os.path.exists(f'Long_Transformer_{method_used}_NRMSE.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  NRMSE_short_results = pd.read_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
  NRMSE_long_results = pd.read_csv(f'Long_Transformer_{method_used}_NRMSE.csv')
  NRMSE_short_results = NRMSE_short_results.iloc[:,1:].values.tolist()
  NRMSE_long_results = NRMSE_long_results.iloc[:,1:].values.tolist()
else:
  NRMSE_short_results = []
  NRMSE_long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    hist, X, y, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      print('Fine Tuning Run ', j+1)
      NRMSE_short_list, NRMSE_long_list = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, j+1, 10, plot_fine_tune)
      print('NRMSE Short Term:', NRMSE_short_list)
      print('NRMSE Long Term:', NRMSE_long_list)
      NRMSE_short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_short_list)
      NRMSE_long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, j+1] + NRMSE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


NRMSE_short_results = pd.DataFrame(NRMSE_short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_long_results = pd.DataFrame(NRMSE_long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list)
NRMSE_short_results.to_csv(f'Short_Transformer_{method_used}_NRMSE.csv')
NRMSE_long_results.to_csv(f'Long_Transformer_{method_used}_NRMSE.csv')

NRMSE_long_results

In [ ]:
NRMSE_short_results[cols_list].median()

In [ ]:
NRMSE_long_results[cols_list].median()